# ViFinQA — Dataset Profile

Notebook này thống kê **chỉ đọc** bộ ViFinQA vừa tải: 1.973 báo cáo tài chính OCR, tập câu hỏi tiếng Việt và danh mục mã cổ phiếu. Kết quả giúp khóa contract cho inventory, ingestion, retrieval và đánh giá chất lượng dữ liệu.

> Chạy **Run All** từ thư mục gốc dự án. Notebook không ghi vào `data/raw`.

## 0. Configuration

Ba nguồn dữ liệu mặc định là `data/raw/code_stock.csv`, `data/raw/questions/questions.jsonl` và `data/raw/financial_statements`. Metadata được quét toàn bộ; nội dung báo cáo chỉ được đọc theo mẫu cố định có giới hạn byte để kiểm soát I/O.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

candidate_roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
PROJECT_ROOT = next(
    (path for path in candidate_roots if (path / 'pyproject.toml').exists()),
    Path.cwd().resolve(),
)
DATA_ROOT = PROJECT_ROOT / 'data' / 'raw'
COMPANY_MAP_PATH = DATA_ROOT / 'code_stock.csv'
QUESTIONS_PATH = DATA_ROOT / 'questions' / 'questions.jsonl'
STATEMENTS_ROOT = DATA_ROOT / 'financial_statements'
CONTENT_SAMPLE_SIZE = 250
RANDOM_SEED = 42
MAX_CONTENT_BYTES = 2_000_000
EXPECTED_YEAR_RANGE = range(2015, 2026)

pd.set_option('display.max_columns', 30)
pd.set_option('display.max_colwidth', 120)
plt.style.use('seaborn-v0_8-whitegrid')

print(f'Project root    : {PROJECT_ROOT}')
print(f'Dataset root    : {DATA_ROOT}')
print(f'Content sample  : at most {CONTENT_SAMPLE_SIZE:,} reports')

In [ ]:
import codecs
import json
import random
import re
from collections.abc import Sequence
from pathlib import Path
from typing import Any

import pandas as pd

COMPANY_COLUMNS = [
    'row_number', 'ticker', 'company_name', 'is_valid', 'validation_issue'
]
QUESTION_COLUMNS = [
    'line_number', 'id', 'question', 'is_valid', 'validation_issue'
]
REPORT_COLUMNS = [
    'path', 'relative_path', 'ticker', 'year', 'year_in_expected_range',
    'document_name', 'statement_type', 'structure_status', 'structure_issue',
    'size_bytes', 'is_empty', 'modified_at', 'stat_error',
]


def load_company_map(path: Path) -> pd.DataFrame:
    """Load the ViFinQA ticker map while retaining invalid rows."""
    frame = pd.read_csv(path, dtype=str, encoding='utf-8-sig').fillna('')
    required = {'Mã CK', 'Tên công ty'}
    if not required <= set(frame.columns):
        raise ValueError(f'Company map must contain columns: {sorted(required)}')
    result = pd.DataFrame(
        {
            'row_number': range(2, len(frame) + 2),
            'ticker': frame['Mã CK'].str.strip().str.upper(),
            'company_name': frame['Tên công ty'].str.strip(),
        }
    )
    valid_ticker = result['ticker'].str.fullmatch(r'[A-Z0-9]{2,10}')
    result['is_valid'] = valid_ticker & result['company_name'].ne('')
    result['validation_issue'] = None
    result.loc[~valid_ticker, 'validation_issue'] = 'invalid or missing ticker'
    result.loc[
        valid_ticker & result['company_name'].eq(''), 'validation_issue'
    ] = 'missing company name'
    return result.reindex(columns=COMPANY_COLUMNS)


def load_questions(path: Path) -> pd.DataFrame:
    """Parse JSON Lines one row at a time and preserve validation failures."""
    records: list[dict[str, object]] = []
    with path.open(encoding='utf-8') as stream:
        for line_number, raw_line in enumerate(stream, start=1):
            record: dict[str, object] = {
                'line_number': line_number,
                'id': None,
                'question': None,
                'is_valid': False,
                'validation_issue': None,
            }
            try:
                payload = json.loads(raw_line)
            except json.JSONDecodeError as error:
                record['validation_issue'] = f'invalid JSON: {error.msg}'
            else:
                if isinstance(payload, dict):
                    record['id'] = payload.get('id')
                    record['question'] = payload.get('question')
                issues = []
                if not isinstance(record['id'], int):
                    issues.append('id must be an integer')
                question = record['question']
                if not isinstance(question, str) or not question.strip():
                    issues.append('question must be a non-empty string')
                record['is_valid'] = not issues
                record['validation_issue'] = '; '.join(issues) or None
            records.append(record)
    return pd.DataFrame.from_records(records, columns=QUESTION_COLUMNS)


def parse_report_path(path: Path, root: Path) -> dict[str, object]:
    """Parse TICKER/YEAR/DOCUMENT/FILE metadata without dropping anomalies."""
    try:
        relative = path.relative_to(root)
    except ValueError:
        relative = path
    parts = relative.parts
    ticker_raw = parts[0] if len(parts) >= 1 else ''
    year_raw = parts[1] if len(parts) >= 2 else ''
    document_name = parts[2] if len(parts) >= 3 else ''
    ticker = ticker_raw.upper() if re.fullmatch(r'[A-Za-z0-9]{2,10}', ticker_raw) else None
    year = int(year_raw) if re.fullmatch(r'\d{4}', year_raw) else None
    normalized_name = document_name.casefold()
    if 'consolidated' in normalized_name:
        statement_type = 'consolidated'
    elif 'separate' in normalized_name:
        statement_type = 'separate'
    elif 'aggregated' in normalized_name:
        statement_type = 'aggregated'
    else:
        statement_type = 'other'

    issues = []
    if len(parts) != 4:
        issues.append('expected exactly ticker/year/document/file hierarchy')
    if ticker is None:
        issues.append('invalid ticker directory')
    if year is None or not 1900 <= year <= 2100:
        issues.append('invalid year directory')

    return {
        'path': path,
        'relative_path': relative.as_posix(),
        'ticker': ticker,
        'year': year,
        'year_in_expected_range': year in range(2015, 2026) if year is not None else False,
        'document_name': document_name or None,
        'statement_type': statement_type,
        'structure_status': 'valid' if not issues else 'malformed',
        'structure_issue': '; '.join(issues) or None,
    }


def build_report_inventory(root: Path) -> pd.DataFrame:
    """Collect deterministic metadata for every TXT report below root."""
    if not root.exists():
        raise FileNotFoundError(f'Statements root does not exist: {root}')
    records: list[dict[str, Any]] = []
    paths = sorted(root.rglob('*.txt'), key=lambda item: item.relative_to(root).as_posix())
    for path in paths:
        record: dict[str, Any] = parse_report_path(path, root)
        try:
            stat = path.stat()
            record.update(
                size_bytes=stat.st_size,
                is_empty=stat.st_size == 0,
                modified_at=pd.Timestamp(stat.st_mtime, unit='s', tz='UTC'),
                stat_error=None,
            )
        except OSError as error:
            record.update(
                size_bytes=None,
                is_empty=False,
                modified_at=pd.NaT,
                stat_error=str(error),
            )
        records.append(record)
    return pd.DataFrame.from_records(records, columns=REPORT_COLUMNS)


def extract_mentioned_tickers(
    question: str, known_tickers: Sequence[str]
) -> tuple[str, ...]:
    """Extract known ticker tokens without matching ticker substrings."""
    matches = []
    for ticker in sorted({value.upper() for value in known_tickers}):
        pattern = rf'(?<![A-Z0-9]){re.escape(ticker)}(?![A-Z0-9])'
        if re.search(pattern, question.upper()):
            matches.append(ticker)
    return tuple(matches)


def sample_paths(paths: Sequence[Path], sample_size: int, seed: int) -> list[Path]:
    """Return a stable bounded sample independent of input ordering."""
    if sample_size < 0:
        raise ValueError('sample_size must not be negative')
    ordered = sorted((Path(path) for path in paths), key=lambda item: item.as_posix().casefold())
    if sample_size >= len(ordered):
        return ordered
    return sorted(
        random.Random(seed).sample(ordered, sample_size),
        key=lambda item: item.as_posix().casefold(),
    )


def inspect_text_file(path: Path, max_bytes: int) -> dict[str, object]:
    """Inspect bounded bytes and retain decoding/read failures as metrics."""
    if max_bytes < 1:
        raise ValueError('max_bytes must be at least 1')
    result: dict[str, object] = {
        'path': path,
        'bytes_read': 0,
        'truncated': False,
        'utf8_valid': None,
        'read_error': None,
        'line_count': 0,
        'character_count': 0,
        'replacement_char_count': 0,
        'replacement_ratio': 0.0,
        'control_char_count': 0,
        'numeric_ratio': 0.0,
        'has_html_table': False,
        'has_pipe_table': False,
        'has_tabular_markers': False,
    }
    try:
        with path.open('rb') as stream:
            payload = stream.read(max_bytes + 1)
    except OSError as error:
        result['read_error'] = str(error)
        return result

    truncated = len(payload) > max_bytes
    payload = payload[:max_bytes]
    try:
        decoder = codecs.getincrementaldecoder('utf-8')(errors='strict')
        text = decoder.decode(payload, final=not truncated)
        utf8_valid = True
    except UnicodeDecodeError:
        decoder = codecs.getincrementaldecoder('utf-8')(errors='replace')
        text = decoder.decode(payload, final=not truncated)
        utf8_valid = False

    lines = text.splitlines()
    lowered = text.casefold()
    non_whitespace = [character for character in text if not character.isspace()]
    replacement_count = text.count('�')
    html_table = '<table' in lowered or ('<tr' in lowered and '<td' in lowered)
    pipe_table = any(line.count('|') >= 2 for line in lines)
    result.update(
        bytes_read=len(payload),
        truncated=truncated,
        utf8_valid=utf8_valid,
        line_count=len(lines),
        character_count=len(text),
        replacement_char_count=replacement_count,
        replacement_ratio=replacement_count / max(len(text), 1),
        control_char_count=sum(
            ord(character) < 32 and character not in '\n\r\t' for character in text
        ),
        numeric_ratio=sum(character.isdigit() for character in non_whitespace)
        / max(len(non_whitespace), 1),
        has_html_table=html_table,
        has_pipe_table=pipe_table,
        has_tabular_markers=html_table or pipe_table or '\t' in text,
    )
    return result

## 1. Synthetic self-check

Kiểm tra nhanh các helper bằng dữ liệu tạm, không ghi vào snapshot thật.

In [ ]:
import tempfile

with tempfile.TemporaryDirectory() as temporary_directory:
    temporary_root = Path(temporary_directory)
    statement_root = temporary_root / 'financial_statements'
    report = (
        statement_root
        / 'AAA'
        / '2015'
        / 'AAA_financial_statements_2015_consolidated'
        / 'report.txt'
    )
    report.parent.mkdir(parents=True)
    report.write_text('<table><tr><td>100</td></tr></table>', encoding='utf-8')
    company_file = temporary_root / 'code_stock.csv'
    company_file.write_text(
        'Mã CK,Tên công ty\nAAA,Công ty AAA\n', encoding='utf-8'
    )
    question_file = temporary_root / 'questions.jsonl'
    question_file.write_text(
        '{"id": 1, "question": "Doanh thu AAA năm 2015?"}\n',
        encoding='utf-8',
    )
    assert load_company_map(company_file)['is_valid'].all()
    assert load_questions(question_file)['is_valid'].all()
    assert build_report_inventory(statement_root).loc[0, 'statement_type'] == 'consolidated'
    assert extract_mentioned_tickers('Doanh thu AAA?', ['AAA']) == ('AAA',)
    assert inspect_text_file(report, 10_000)['has_html_table'] is True

print('Synthetic self-check passed')

## 2. Dataset overview

Nạp ba nguồn ViFinQA, giữ riêng từng inventory và hiển thị các KPI cấp phát hành.

In [ ]:
required_paths = [COMPANY_MAP_PATH, QUESTIONS_PATH, STATEMENTS_ROOT]
missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    formatted = '\n'.join(f'- {path}' for path in missing_paths)
    raise FileNotFoundError(f'Missing ViFinQA inputs:\n{formatted}')

company_map = load_company_map(COMPANY_MAP_PATH)
questions = load_questions(QUESTIONS_PATH)
report_inventory = build_report_inventory(STATEMENTS_ROOT)
valid_companies = company_map[company_map['is_valid']].copy()
valid_questions = questions[questions['is_valid']].copy()
valid_reports = report_inventory[
    report_inventory['structure_status'].eq('valid')
].copy()

for label, frame in (
    ('company map', valid_companies),
    ('questions', valid_questions),
    ('reports', valid_reports),
):
    if frame.empty:
        raise RuntimeError(f'No valid {label} records were found in {DATA_ROOT}')

In [ ]:
valid_years = valid_reports['year'].dropna().astype(int)
kpis = pd.DataFrame(
    [
        ('Question rows', f'{len(questions):,}'),
        ('Valid questions', f'{len(valid_questions):,}'),
        ('Report files', f'{len(report_inventory):,}'),
        ('Mapped companies', f'{len(valid_companies):,}'),
        ('Observed report tickers', f"{valid_reports['ticker'].nunique():,}"),
        (
            'Observed years',
            f'{valid_years.min()}–{valid_years.max()}' if len(valid_years) else 'n/a',
        ),
        (
            'Report text size',
            f"{report_inventory['size_bytes'].fillna(0).sum() / 1024**2:,.1f} MiB",
        ),
        (
            'Malformed report paths',
            f"{report_inventory['structure_status'].eq('malformed').sum():,}",
        ),
        ('Empty reports', f"{report_inventory['is_empty'].fillna(False).sum():,}"),
        ('Invalid question rows', f"{(~questions['is_valid']).sum():,}"),
        ('Invalid company rows', f"{(~company_map['is_valid']).sum():,}"),
    ],
    columns=['Metric', 'Value'],
)
display(kpis.style.hide(axis='index').set_properties(**{'text-align': 'left'}))

## 3. Report coverage

Độ phủ báo cáo theo năm, mã cổ phiếu và loại báo cáo. Ô trống trên heatmap chỉ có nghĩa là không có file tương ứng, không có nghĩa giá trị tài chính bằng 0.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
reports_by_year = valid_reports['year'].value_counts().sort_index()
reports_by_year.plot(kind='bar', ax=axes[0, 0], color='#2563eb', title='Reports by year')
axes[0, 0].set(xlabel='Year', ylabel='Reports')

top_tickers = valid_reports['ticker'].value_counts().head(20).sort_values()
top_tickers.plot(kind='barh', ax=axes[0, 1], color='#0f766e', title='Top 20 tickers')
axes[0, 1].set(xlabel='Reports', ylabel='Ticker')

statement_types = valid_reports['statement_type'].value_counts().sort_values()
statement_types.plot(
    kind='barh', ax=axes[1, 0], color='#7c3aed', title='Statement types'
)
axes[1, 0].set(xlabel='Reports', ylabel='Type')

positive_sizes_mib = (
    report_inventory.loc[report_inventory['size_bytes'].gt(0), 'size_bytes'] / 1024**2
)
axes[1, 1].hist(positive_sizes_mib, bins=45, color='#ea580c', alpha=0.85)
axes[1, 1].set(title='Report size distribution', xlabel='MiB', ylabel='Reports')
fig.suptitle('ViFinQA report corpus', fontsize=16, fontweight='bold')
fig.tight_layout()
plt.show()

In [ ]:
coverage_tickers = valid_reports['ticker'].value_counts().head(40).index
coverage = (
    valid_reports[valid_reports['ticker'].isin(coverage_tickers)]
    .pivot_table(
        index='ticker',
        columns='year',
        values='relative_path',
        aggfunc='size',
        fill_value=0,
    )
    .reindex(
        index=coverage_tickers,
        columns=list(EXPECTED_YEAR_RANGE),
        fill_value=0,
    )
)
fig, ax = plt.subplots(figsize=(15, max(6, len(coverage) * 0.28)))
image = ax.imshow(coverage.to_numpy(), aspect='auto', cmap='Blues')
ax.set_xticks(
    range(len(coverage.columns)),
    labels=[str(year) for year in coverage.columns],
    rotation=45,
)
ax.set_yticks(range(len(coverage.index)), labels=coverage.index)
ax.set(title='Ticker–year report coverage', xlabel='Year', ylabel='Ticker')
fig.colorbar(image, ax=ax, label='Reports')
fig.tight_layout()
plt.show()

coverage_summary = pd.DataFrame(
    {
        'reports': valid_reports.groupby('ticker').size(),
        'years_present': valid_reports.groupby('ticker')['year'].nunique(),
        'first_year': valid_reports.groupby('ticker')['year'].min(),
        'last_year': valid_reports.groupby('ticker')['year'].max(),
    }
).sort_values(['years_present', 'reports'], ascending=False)
display(coverage_summary.head(50))

## 4. Question analysis

Phân tích độ dài, ID, câu trùng, năm và mã cổ phiếu được nhắc tới. Việc nhận diện ticker chỉ dùng token đã có trong `code_stock.csv`, nên câu viết hoàn toàn bằng tên công ty có thể được ghi nhận là chưa có ticker.

In [ ]:
known_tickers = valid_companies['ticker'].tolist()
question_profile = valid_questions.copy()
question_profile['character_count'] = question_profile['question'].str.len()
question_profile['word_count'] = question_profile['question'].str.split().str.len()
question_profile['mentioned_years'] = question_profile['question'].map(
    lambda value: tuple(sorted(set(re.findall(r'\b(?:19|20)\d{2}\b', value))))
)
question_profile['mentioned_tickers'] = question_profile['question'].map(
    lambda value: extract_mentioned_tickers(value, known_tickers)
)
numeric_terms = (
    'bao nhiêu', 'tỷ lệ', 'chênh lệch', 'tăng', 'giảm',
    'trung bình', 'tổng', 'phần trăm',
)
question_profile['has_numeric_language'] = question_profile['question'].map(
    lambda value: bool(re.search(r'\d', value))
    or any(term in value.casefold() for term in numeric_terms)
)

duplicate_ids = question_profile[
    question_profile.duplicated('id', keep=False)
].sort_values('id')
duplicate_text = question_profile[
    question_profile.duplicated('question', keep=False)
].sort_values('question')
observed_ids = set(question_profile['id'].astype(int))
missing_ids = (
    sorted(set(range(1, max(observed_ids) + 1)) - observed_ids)
    if observed_ids
    else []
)
mentioned_year_counts = (
    question_profile['mentioned_years'].explode().dropna().value_counts().sort_index()
)
mentioned_ticker_counts = (
    question_profile['mentioned_tickers'].explode().dropna().value_counts().head(20)
)

question_kpis = pd.DataFrame(
    [
        ('Valid questions', f'{len(question_profile):,}'),
        ('Duplicate ID rows', f'{len(duplicate_ids):,}'),
        ('Duplicate text rows', f'{len(duplicate_text):,}'),
        ('Missing IDs', f'{len(missing_ids):,}'),
        (
            'Questions with recognized ticker',
            f"{question_profile['mentioned_tickers'].str.len().gt(0).mean():.1%}",
        ),
        (
            'Questions with numeric language',
            f"{question_profile['has_numeric_language'].mean():.1%}",
        ),
        ('Median words', f"{question_profile['word_count'].median():,.0f}"),
    ],
    columns=['Metric', 'Value'],
)
display(question_kpis.style.hide(axis='index'))
display(question_profile[['character_count', 'word_count']].describe().round(1))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].hist(question_profile['word_count'], bins=30, color='#2563eb')
axes[0].set(title='Question length', xlabel='Whitespace-separated words', ylabel='Questions')
mentioned_year_counts.plot(kind='bar', ax=axes[1], color='#0f766e', title='Mentioned years')
axes[1].set(xlabel='Year', ylabel='Questions')
mentioned_ticker_counts.sort_values().plot(
    kind='barh', ax=axes[2], color='#7c3aed', title='Top mentioned tickers'
)
axes[2].set(xlabel='Questions', ylabel='Ticker')
fig.tight_layout()
plt.show()

if len(duplicate_ids):
    display(Markdown('### Duplicate question IDs'))
    display(duplicate_ids.head(30))
if len(duplicate_text):
    display(Markdown('### Duplicate question text'))
    display(duplicate_text.head(30))
if missing_ids:
    display(Markdown(f'**Missing IDs:** `{missing_ids[:50]}`'))

## 5. Integrity checks

Đối chiếu câu hỏi, đường dẫn báo cáo và danh mục công ty. Các bảng chi tiết là tín hiệu cần xử lý, không tự động xóa hay sửa raw data.

In [ ]:
company_tickers = set(valid_companies['ticker'])
report_tickers = set(valid_reports['ticker'].dropna())
question_tickers = set(question_profile['mentioned_tickers'].explode().dropna())
report_tickers_not_mapped = sorted(report_tickers - company_tickers)
mapped_tickers_without_reports = sorted(company_tickers - report_tickers)
mentioned_tickers_without_reports = sorted(question_tickers - report_tickers)
questions_without_ticker = question_profile[
    question_profile['mentioned_tickers'].str.len().eq(0)
]
invalid_questions = questions[~questions['is_valid']]
malformed_reports = report_inventory[
    report_inventory['structure_status'].eq('malformed')
]
duplicate_company_tickers = valid_companies[
    valid_companies.duplicated('ticker', keep=False)
].sort_values('ticker')

integrity_summary = pd.DataFrame(
    [
        ('Report tickers absent from map', len(report_tickers_not_mapped)),
        ('Mapped tickers without reports', len(mapped_tickers_without_reports)),
        ('Mentioned tickers without reports', len(mentioned_tickers_without_reports)),
        ('Questions without recognized ticker', len(questions_without_ticker)),
        ('Invalid question rows', len(invalid_questions)),
        ('Malformed report paths', len(malformed_reports)),
        ('Duplicate company-map rows', len(duplicate_company_tickers)),
    ],
    columns=['Check', 'Count'],
)
display(integrity_summary.style.hide(axis='index').background_gradient(
    subset=['Count'], cmap='Oranges'
))

for title, values in (
    ('Report tickers absent from company map', report_tickers_not_mapped),
    ('Mapped tickers without reports', mapped_tickers_without_reports),
    ('Mentioned tickers without reports', mentioned_tickers_without_reports),
):
    if values:
        display(Markdown(f'### {title}'))
        display(pd.DataFrame({'ticker': values}).head(50))

if len(invalid_questions):
    display(Markdown('### Invalid question rows'))
    display(invalid_questions.head(30))
if len(malformed_reports):
    display(Markdown('### Malformed report paths'))
    display(malformed_reports.head(30))

## 6. Report content sample

Đọc tối đa `MAX_CONTENT_BYTES` từ một mẫu xác định bởi `RANDOM_SEED`. Các tỷ lệ dưới đây mô tả mẫu, không phải kiểm định toàn bộ corpus.

In [ ]:
candidate_paths = valid_reports.loc[valid_reports['stat_error'].isna(), 'path'].tolist()
selected_paths = sample_paths(candidate_paths, CONTENT_SAMPLE_SIZE, RANDOM_SEED)
content_profile = pd.DataFrame(
    [inspect_text_file(path, MAX_CONTENT_BYTES) for path in selected_paths]
)
if content_profile.empty:
    raise RuntimeError('No readable report candidates are available for content sampling.')

readable_content = content_profile[content_profile['read_error'].isna()]
strict_utf8_share = (
    float(readable_content['utf8_valid'].eq(True).mean())
    if len(readable_content)
    else None
)
content_kpis = pd.DataFrame(
    [
        ('Sampled reports', f'{len(content_profile):,}'),
        ('Successfully read', f'{len(readable_content):,}'),
        ('Read errors', f"{content_profile['read_error'].notna().sum():,}"),
        (
            'Strict UTF-8 among readable reports',
            f'{strict_utf8_share:.1%}' if strict_utf8_share is not None else 'n/a',
        ),
        ('Truncated at byte cap', f"{content_profile['truncated'].mean():.1%}"),
        ('HTML table markers', f"{content_profile['has_html_table'].mean():.1%}"),
        ('Any simple table marker', f"{content_profile['has_tabular_markers'].mean():.1%}"),
        ('Median lines', f"{content_profile['line_count'].median():,.0f}"),
        ('Median numeric ratio', f"{content_profile['numeric_ratio'].median():.1%}"),
    ],
    columns=['Metric', 'Value'],
)
display(content_kpis.style.hide(axis='index'))

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
axes[0].hist(content_profile['line_count'], bins=40, color='#2563eb')
axes[0].set(title='Line count in sample', xlabel='Lines', ylabel='Reports')
axes[1].hist(content_profile['numeric_ratio'], bins=30, color='#0f766e')
axes[1].set(title='Numeric-character ratio', xlabel='Ratio', ylabel='Reports')
marker_rates = content_profile[
    ['has_html_table', 'has_pipe_table', 'has_tabular_markers']
].mean().sort_values()
marker_rates.plot(kind='barh', ax=axes[2], color='#7c3aed', title='Table markers')
axes[2].set(xlabel='Share of sample', xlim=(0, 1))
fig.tight_layout()
plt.show()

display(Markdown('### Highest replacement-character ratios'))
display(content_profile.sort_values('replacement_ratio', ascending=False).head(30))

## 7. Readiness summary

Chuyển các quan sát thành hành động ưu tiên cho inventory, ingestion và retrieval. Evidence được tính trực tiếp từ snapshot hiện tại.

In [ ]:
readiness = pd.DataFrame(
    [
        (
            'P0',
            'question validation',
            f'{len(invalid_questions):,} invalid rows',
            'Quarantine invalid JSONL rows while retaining line-number provenance.',
        ),
        (
            'P0',
            'question IDs',
            f'{len(duplicate_ids):,} duplicate rows; {len(missing_ids):,} missing IDs',
            'Use validated question IDs as stable external identifiers.',
        ),
        (
            'P0',
            'report paths',
            f'{len(malformed_reports):,} malformed paths',
            'Keep tolerant parsing and record structure issues in the inventory.',
        ),
        (
            'P1',
            'ticker alignment',
            f'{len(report_tickers_not_mapped):,} unmapped report tickers; '
            f'{len(mapped_tickers_without_reports):,} mapped without reports',
            'Resolve ticker mismatches before retrieval indexing.',
        ),
        (
            'P1',
            'encoding',
            f"{content_profile['utf8_valid'].eq(False).sum():,} sampled non-UTF-8 reports",
            'Preserve raw bytes and record decoder fallbacks.',
        ),
        (
            'P1',
            'table handling',
            f"{content_profile['has_tabular_markers'].mean():.1%} sampled with table markers",
            'Retain inline tables and add structure-aware chunking.',
        ),
    ],
    columns=['priority', 'finding', 'evidence', 'next_action'],
)
display(
    readiness.style.hide(axis='index')
    .set_properties(subset=['evidence'], **{'text-align': 'left', 'min-width': '220px'})
    .set_properties(subset=['next_action'], **{'text-align': 'left', 'min-width': '420px'})
)

print(
    'Suggested order: validated inventory → tolerant TXT reader → provenance → '
    'table-aware chunking → retrieval baseline.'
)